# 03 · Isolation Forest anomaly detection

Flags anomalous trust-years from the structured data (expected vs actual cost, variance, and the audit outcome flags). The Isolation Forest is implemented from scratch; `contamination = 0.315` sets the share of trust-years flagged.

In [1]:
import numpy as np
import pandas as pd
import random
from sklearn.metrics import confusion_matrix

# fixed seeds so the anomaly flags are reproducible
random.seed(42)
np.random.seed(42)

df_trusts = pd.read_excel("../data/finaldatasetdissertation.xlsx")

# Binary flags from the audit outcome columns
df_trusts["OpinionOnFinancialStatements"] = df_trusts["OpinionOnFinancialStatements"].fillna("standard").str.strip().str.lower()
df_trusts["Opinion_NonStandard"] = df_trusts["OpinionOnFinancialStatements"].eq("non-standard").astype(int)

df_trusts["AdditionalReportingPowers"] = df_trusts["AdditionalReportingPowers"].fillna("not exercised").str.strip().str.lower()
df_trusts["Exercised"] = df_trusts["AdditionalReportingPowers"].eq("exercised").astype(int)

df_trusts["VfmArrangementsConclusion"] = df_trusts["VfmArrangementsConclusion"].fillna("qualified").str.strip().str.lower()
# compare with lowercase "qualified" (the column has just been lowercased)
df_trusts["Qualified"] = df_trusts["VfmArrangementsConclusion"].eq("qualified").astype(int)

df_trusts[["OpinionOnFinancialStatements", "Opinion_NonStandard",
           "AdditionalReportingPowers", "Exercised",
           "VfmArrangementsConclusion", "Qualified"]].head()

,OpinionOnFinancialStatements,Opinion_NonStandard,AdditionalReportingPowers,Exercised,VfmArrangementsConclusion,Qualified
0,standard,0,not exerised,0,unqualified,0
1,standard,0,not exerised,0,unqualified,0
2,standard,0,exercised,1,unqualified,0
3,standard,0,not exerised,0,unqualified,0
4,non-standard,1,exercised,1,qualified,1


In [2]:
# Isolation Forest implemented from scratch, adapted from
# https://github.com/jyotipmahes/Isolation-Forest-for-Anomaly-Detection/blob/master/iforest.py

class IsolationTreeEnsemble:
    def __init__(self, sample_size, n_trees=10):
        self.sample_size = sample_size
        self.n_trees = n_trees

    def fit(self, X:np.ndarray, improved=False):
        """Implementation you already have..."""
        if isinstance(X, pd.DataFrame):
            X = X.values
        self.height_limit = np.ceil(np.log2(self.sample_size))
        self.trees = []
        for i in range(self.n_trees):
            X_sample = X[random.sample(range(X.shape[0]), self.sample_size)]
            t = IsolationTree(self.height_limit)
            t.root = t.fit(X_sample, improved)
            self.trees.append(t)
        return self

    def path_length(self, X:np.ndarray) -> np.ndarray:
        """Implementation you already have..."""
        path_matrix = np.zeros((X.shape[0], self.n_trees))
        indexes = np.array([True] * X.shape[0])
        for j, t in enumerate(self.trees):
            currHeight = 0
            pathLength(path_matrix, indexes, X, j, t.root, currHeight)
        return np.mean(path_matrix, axis=1)

    def anomaly_score(self, X:np.ndarray) -> np.ndarray:
        """Implementation you already have..."""
        if isinstance(X, pd.DataFrame):
            X = X.values
        avg_path = self.path_length(X)
        score = 2**(-(avg_path/c(self.sample_size)))
        return score

    def predict(self, X: np.ndarray | pd.DataFrame, threshold: float | None = None, contamination: float = 0.315) -> np.ndarray:
        """Implementation you already have..."""
        if isinstance(X, pd.DataFrame):
            X = X.values
        scores = self.anomaly_score(X)
        if threshold is not None:
            return (scores >= threshold).astype(int)
        else:
            k = int(len(scores) * contamination)
            if k == 0:
                return np.zeros_like(scores, dtype=int)
            threshold = np.partition(scores, -k)[-k]
            return (scores >= threshold).astype(int)
        return self
    
class IsolationTree:
    def __init__(self, height_limit):
        self.height_limit = height_limit
        self.height = 0
        self.root = None

    def fit(self, X:np.ndarray, improved=False):
        """
        Given a 2D matrix of observations, create an isolation tree. Set field
        self.root to the root of that tree and return it.

        If you are working on an improved algorithm, check parameter "improved"
        and switch to your new functionality else fall back on your original code.
        """
        if not improved:
            n_nodes =1
            self.root, n_nodes = fit2(X, self.height, self.height_limit, n_nodes)
            self.n_nodes = n_nodes
        else:
            n_nodes =1
            self.root, n_nodes = fit_improved(X, self.height, self.height_limit, n_nodes)
            self.n_nodes = n_nodes
        return self.root

def find_TPR_threshold(y, scores, desired_TPR):
    """
    Start at score threshold 1.0 and work down until we hit desired TPR.
    Step by 0.01 score increments. For each threshold, compute the TPR
    and FPR to see if we've reached to the desired TPR. If so, return the
    score threshold and FPR.
    """
    for threshold in np.arange(1,0,-0.001):
        y_pred = (scores>=threshold)*1
        confusion = confusion_matrix(y, y_pred)
        TN, FP, FN, TP = confusion.flat
        TPR = TP / (TP + FN)
        FPR = FP / (FP + TN)
        if desired_TPR*.95<TPR<desired_TPR*1.02:
            return threshold, FPR


def fit_improved(X: np.ndarray, height, limit, n_nodes):

    if height >= limit or len(X) <2:
        return exNode(len(X)), n_nodes


    else:
        rows, cols = X.shape
        good_split = True

        while good_split:
            choosen_col = np.random.randint(0,cols)
            mx = X[:, choosen_col].max()
            mn = X[:, choosen_col].min()

            if mx==mn:
                return exNode(len(X)), n_nodes


            split = np.random.uniform(mn, mx)
            Xl = X[X[:, choosen_col] < split]
            Xr = X[X[:, choosen_col] >= split]


            if len(Xl)<=0.15*len(Xr) or len(Xr)<0.15*len(Xl) or len(X)<12:
                good_split=False
        left, n_nodes = fit2(Xl, height + 1, limit, n_nodes+1)
        right, n_nodes = fit2(Xr, height + 1, limit, n_nodes+1)
        return inNode(left,
                          right,
                          choosen_col,
                          split), n_nodes


def fit2(X: np.ndarray, height, limit, n_nodes):

    if height >= limit or len(X) <2:
        return exNode(len(X)), n_nodes


    else:
        rows, cols = X.shape
        good_split = True


        choosen_col = np.random.randint(0,cols)
        mx = X[:, choosen_col].max()
        mn = X[:, choosen_col].min()

        if mx==mn:
            return exNode(len(X)), n_nodes


        split = np.random.uniform(mn, mx)
        Xl = X[X[:, choosen_col] < split]
        Xr = X[X[:, choosen_col] >= split]
        left, n_nodes = fit2(Xl, height + 1, limit, n_nodes+1)
        right, n_nodes = fit2(Xr, height + 1, limit, n_nodes+1)
        return inNode(left,
                          right,
                          choosen_col,
                          split), n_nodes


def H(i):
    return np.log(i) + 0.5772156649


def c(size):
    if size > 2:
        return 2 * H(size - 1) - 2 * (size - 1.0) / size
    elif size == 2:
        return 1
    else:
        return 0
    
class exNode:
    def __init__(self, size):
        self.size = size


class inNode:
    def __init__(self, left, right, splitAtt, splitValue):
        self.left = left
        self.right = right
        self.splitAtt = splitAtt
        self.splitValue = splitValue


def pathLength(path_matrix,obs_indexes,x,tree_index, Tree,currHeight):
    if isinstance(Tree,exNode):
        path_matrix[obs_indexes, tree_index] = currHeight+c(Tree.size)
    else:
        a=Tree.splitAtt
        left_indexes = (obs_indexes)*(x[:,a]<Tree.splitValue)
        pathLength(path_matrix, left_indexes,x,tree_index,Tree.left,currHeight+1)
        right_indexes = (obs_indexes)*(x[:,a]>=Tree.splitValue)
        pathLength(path_matrix, right_indexes,x,tree_index,Tree.right,currHeight+1)


In [3]:
cols_to_use = [
    "Sum of Expected Cost (£)",
    "Sum of Actual Cost (£)",
    "Sum of Variance (£)",
    "Actual to expected",
    "Opinion_NonStandard",
    "Exercised",
    "Qualified"
]

features = df_trusts[cols_to_use].dropna()
model = IsolationTreeEnsemble(sample_size=256).fit(features.values)
predictions = model.predict(features.values)

# assign by index so rows line up even if some were dropped
df_trusts.loc[features.index, "iForest"] = predictions
df_trusts["iForest"].value_counts()

iForest
0.0    681
1.0    313
Name: count, dtype: int64

In [4]:
df_trusts.to_excel("../data/dataset_with_iforest.xlsx", index=False)